# Advanced Analytics: Customer 360 Analytical Mart (`gold.report_customers`)

## Executive Summary & Architecture
This notebook builds and validates the **Customer 360 Analytical Data Mart** (`gold.report_customers`). Consolidating customer demographics, purchasing behavior, lifetime order statistics, and recency into a unified analytical view empowers:
- Customer Success and Retention teams with customer-level behavioral profiles.
- Marketing analytics teams with pre-aggregated RFM segments and age cohorts.
- BI reporting systems with high-performance dimensional metrics.

## Schema Specification (15 Columns)
1. `customer_key`: Surrogate key from `gold.dim_customers`.
2. `customer_number`: Natural business identifier.
3. `customer_name`: Full concatenated name (`first_name + ' ' + last_name`).
4. `age`: Customer age in years.
5. `age_group`: Demographic bracket (`'Under 20'`, `'20-29'`, `'30-39'`, `'40-49'`, `'50 and above'`).
6. `customer_segment`: RFM tenure tier (`'VIP'`, `'Regular'`, `'New'`).
7. `last_order_date`: Most recent transaction date.
8. `recency`: Months elapsed between last order date and current date.
9. `total_orders`: Count of distinct purchase orders.
10. `total_sales`: Lifetime aggregate spending.
11. `total_quantity`: Cumulative item count purchased.
12. `total_products`: Count of unique SKUs ordered.
13. `lifespan`: Duration in months between first and last purchase.
14. `avg_order_value`: Average order value (AOV = `total_sales / total_orders`).
15. `avg_monthly_spend`: Average expenditure per active month (`total_sales / lifespan`).

## Dialect Translation & Safety Rules
- **Recency Calculation**: Computes customer recency metrics using Spark SQL `months_between()`:
  - Age: `cast(floor(months_between(current_date(), c.birthdate) / 12) as int)`
  - Lifespan: `cast(floor(months_between(max(order_date), min(order_date))) as int)`
  - Recency: `cast(floor(months_between(current_date(), last_order_date)) as int)`
- **Division-by-Zero Protection**: Guarded `avg_order_value` and `avg_monthly_spend` using explicit CASE expressions.

## Data Contracts & Layer Dependencies
- **Target View Created**: `gold.report_customers`
- **Source Tables Referenced**:
  - `gold.fact_sales` (`order_number`, `product_key`, `order_date`, `sales_amount`, `quantity`, `customer_key`)
  - `gold.dim_customers` (`customer_key`, `customer_number`, `first_name`, `last_name`, `birthdate`)

In [ ]:
# Databricks notebook source
# COMMAND ----------
# Setup display helper and session verification
try:
    display
except NameError:
    def display(df, n=20):
        """Fallback display function for standard PySpark and Python environments."""
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)

# Ensure gold database context
spark.sql("USE gold")
print("Session configured. Active database context: gold")

## 1. DDL & Mart Transformation: Create `gold.report_customers`

### Transformation Pipeline
1. `base_query`: Joins sales facts to customer demographics, filtering for valid sales dates.
2. `customer_aggregation`: Computes order counts, total revenue, unit volumes, and relationship lifespan.
3. `final_projection`: Appends RFM segmentations, demographic age brackets, recency, AOV, and average monthly spend.

In [ ]:
# COMMAND ----------
# 1. Create or Replace View: gold.report_customers
spark.sql("""
CREATE OR REPLACE VIEW gold.report_customers AS
WITH base_query AS (
    SELECT
        f.order_number,
        f.product_key,
        f.order_date,
        f.sales_amount,
        f.quantity,
        c.customer_key,
        c.customer_number,
        concat(c.first_name, ' ', c.last_name) AS customer_name,
        cast(floor(months_between(current_date(), c.birthdate) / 12) as int) AS age
    FROM gold.fact_sales f
    LEFT JOIN gold.dim_customers c
        ON c.customer_key = f.customer_key
    WHERE f.order_date IS NOT NULL
),
customer_aggregation AS (
    SELECT 
        customer_key,
        customer_number,
        customer_name,
        age,
        count(distinct order_number) AS total_orders,
        sum(sales_amount) AS total_sales,
        sum(quantity) AS total_quantity,
        count(distinct product_key) AS total_products,
        max(order_date) AS last_order_date,
        cast(floor(months_between(max(order_date), min(order_date))) as int) AS lifespan
    FROM base_query
    GROUP BY customer_key, customer_number, customer_name, age
)
SELECT
    customer_key,
    customer_number,
    customer_name,
    age,
    CASE
        WHEN age < 20 THEN 'Under 20'
        WHEN age BETWEEN 20 AND 29 THEN '20-29'
        WHEN age BETWEEN 30 AND 39 THEN '30-39'
        WHEN age BETWEEN 40 AND 49 THEN '40-49'
        ELSE '50 and above'
    END AS age_group,
    CASE
        WHEN lifespan >= 12 AND total_sales > 5000 THEN 'VIP'
        WHEN lifespan >= 12 AND total_sales <= 5000 THEN 'Regular'
        ELSE 'New'
    END AS customer_segment,
    last_order_date,
    cast(floor(months_between(current_date(), last_order_date)) as int) AS recency,
    total_orders,
    total_sales,
    total_quantity,
    total_products,
    lifespan,
    CASE
        WHEN total_orders = 0 THEN 0.0
        ELSE cast(total_sales as double) / total_orders 
    END AS avg_order_value,
    CASE
        WHEN lifespan = 0 THEN cast(total_sales as double)
        ELSE round(cast(total_sales as double) / lifespan, 2)
    END AS avg_monthly_spend
FROM customer_aggregation
""")
print("View 'gold.report_customers' created and registered successfully.")

## 2. Customer Segment Profile & Scorecard

### Business Context & Objective
Aggregates the newly constructed `gold.report_customers` view by customer segment to assess commercial impact, account distribution, and average revenue per segment.

In [ ]:
# COMMAND ----------
# 2. Segment Scorecard
df_segment_scorecard = spark.sql("""
SELECT
    customer_segment,
    count(customer_key) AS total_customers,
    sum(total_sales) AS segment_revenue,
    round(avg(total_sales), 2) AS avg_customer_spend,
    round(avg(avg_order_value), 2) AS avg_aov,
    round(avg(lifespan), 1) AS avg_lifespan_months
FROM gold.report_customers
GROUP BY customer_segment
ORDER BY segment_revenue DESC
""")
display(df_segment_scorecard)

## 3. High-Value Customer Inspection (Top 10 Profiles)

### Business Context & Objective
Inspects detailed profiles of the top 10 highest-spending customers for account management.

In [ ]:
# COMMAND ----------
# 3. Top 10 Customer 360 Profiles
df_top_customers = spark.sql("""
SELECT
    customer_number,
    customer_name,
    age,
    age_group,
    customer_segment,
    recency,
    lifespan,
    total_orders,
    total_sales,
    round(avg_order_value, 2) AS avg_order_value,
    round(avg_monthly_spend, 2) AS avg_monthly_spend
FROM gold.report_customers
ORDER BY total_sales DESC
LIMIT 10
""")
display(df_top_customers)

### Strategic Recommendations (Customer 360)
1. **VIP Churn Prevention**: Track `recency` on VIP accounts; trigger proactive outreach if recency exceeds 3 months.
2. **Age-Targeted Merchandising**: Tailor marketing creative to the highest-spending age demographic (e.g. 30-49 age groups).
3. **Cross-Selling**: Focus promotional discounts on high-AOV customers to encourage broader category exploration.